In [ ]:
# ============================================================
# Q3: Retail Store Sales Prediction using Regression Pipeline
# ============================================================

# -----------------------------
# Import Libraries
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# -----------------------------
# 1. Load Dataset
# -----------------------------

df = pd.read_csv("q3_retail_promotions.csv")

print("First Five Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

The dataset contains retail transaction records with information about promotions, store details, competition, and sales performance.

Our target variable is items_sold, and the goal is to predict future sales using regression models.

In [ ]:
# -----------------------------
# 2. Date Feature Engineering
# -----------------------------

# Convert to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract features
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek

# Binary feature: is_month_end
df['is_month_end'] = np.where(
    df['transaction_date'].dt.day >= 25,
    1,
    0
)

# Display sample
print("Sample After Feature Engineering:")
print(df.head())

New time-based features were created because sales patterns often depend on seasonality and calendar effects.

month helps capture monthly sales trends
day_of_week captures weekday vs weekend patterns
is_month_end helps identify end-of-month buying behavior

These features improve model prediction quality.

In [ ]:
# -----------------------------
# 3. Temporal Train-Test Split
# -----------------------------

# Sort by date
df = df.sort_values("transaction_date")

# Define split index (80% train, 20% test)
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print("Training Set Shape:", train_df.shape)
print("Testing Set Shape:", test_df.shape)

A random split is not suitable for time-ordered data because it causes data leakage.

Future records may accidentally appear in the training set while older records appear in testing, which creates unrealistic model evaluation.

Using the most recent 20% as the test set better simulates real-world forecasting where we predict future sales using past data.

In [ ]:
# -----------------------------
# 4. Prepare Features and Target
# -----------------------------

target = "items_sold"

X_train = train_df.drop(columns=[target, "transaction_date"])
y_train = train_df[target]

X_test = test_df.drop(columns=[target, "transaction_date"])
y_test = test_df[target]

# Categorical and Numerical columns
categorical_cols = [
    "promotion_type",
    "location_type",
    "store_size"
]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

In [ ]:
# -----------------------------
# 5. Preprocessing Pipeline
# -----------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numerical_cols
        )
    ]
)

print("Preprocessing Pipeline Created Successfully")

Pipeline Justification

Using a Pipeline ensures preprocessing and model training happen consistently and without data leakage.

The pipeline is fit only on training data and then applied to test data.

This makes the workflow reproducible, clean, and suitable for real-world machine learning systems.

In [ ]:
# -----------------------------
# 6. Model 1 - Linear Regression
# -----------------------------

linear_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_pipeline.fit(X_train, y_train)

linear_preds = linear_pipeline.predict(X_test)

linear_rmse = np.sqrt(mean_squared_error(y_test, linear_preds))
linear_mae = mean_absolute_error(y_test, linear_preds)

print("Linear Regression Results")
print("RMSE:", linear_rmse)
print("MAE:", linear_mae)

In [ ]:
# -----------------------------
# Parity Plot - Linear Regression
# -----------------------------

plt.figure(figsize=(7,6))
plt.scatter(y_test, linear_preds)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)

plt.xlabel("Actual Items Sold")
plt.ylabel("Predicted Items Sold")
plt.title("Parity Plot - Linear Regression")
plt.show()

In [ ]:
# -----------------------------
# 7. Model 2 - Random Forest
# -----------------------------

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        random_state=42,
        n_estimators=100
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_preds = rf_pipeline.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mae = mean_absolute_error(y_test, rf_preds)

print("Random Forest Results")
print("RMSE:", rf_rmse)
print("MAE:", rf_mae)

In [ ]:
# -----------------------------
# Parity Plot - Random Forest
# -----------------------------

plt.figure(figsize=(7,6))
plt.scatter(y_test, rf_preds)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()]
)

plt.xlabel("Actual Items Sold")
plt.ylabel("Predicted Items Sold")
plt.title("Parity Plot - Random Forest")
plt.show()

Model Comparison

RMSE and MAE are used to evaluate regression performance.

MAE shows average prediction error
RMSE penalizes large errors more heavily

The model with lower RMSE and MAE is considered better.

Usually, Random Forest performs better because it captures non-linear relationships more effectively than Linear Regression.

In [ ]:
# -----------------------------
# 8. Feature Importances
# -----------------------------

# Get feature names after one-hot encoding
encoded_cat_features = rf_pipeline.named_steps[
    "preprocessor"
].named_transformers_["cat"].get_feature_names_out(categorical_cols)

all_features = list(encoded_cat_features) + numerical_cols

# Get feature importances
importances = rf_pipeline.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("Top 5 Most Important Features:")
print(feature_importance_df.head(5))

The Random Forest Regressor generally performs better than Linear Regression because it can model complex non-linear relationships in retail sales data.

Feature importance analysis helps identify which factors most strongly affect sales, such as promotions, competition density, and store characteristics.

This pipeline provides a reliable and reproducible system for forecasting retail store sales and improving business decision-making.